In [ ]:
# Satellites in orbit undergo slight orbital drift. To analyze time variations, we must project 
# and interpolate the real observations onto a fixed nominal track.

# In the notebook, we will focus on Pass 50 of Jason-1, which cuts directly across the Gulf Stream 
# in the North-West Atlantic. Our goal is to compute the geostrophic velocity perpendicular to the 
# track (cross-track velocity $v_c$) using the Absolute Dynamic Topography (ADT), which represents 
# the sea surface height above the geoid.

# FOR MORE THEORETICAL BACKGROUND SEE LECTURE L11.1_Analysis_Techniques slides 5-9

In [ ]:
# The data for this notebook were downloaded from NASA PO.DAAC (https://podaac.jpl.nasa.gov/)
# and stored in a more convenient layout in jason1_l2_track_50.csv

# They refer to the following subset:   
# dataset_id="Jason-1 GDR SSHA version E NetCDF",
# track=50
# minimum_longitude=-75,
# maximum_longitude=-65,
# minimum_latitude=30,
# maximum_latitude=42,
# start_datetime="2002-04-26T06:50:50",
# end_datetime="2004-03-31T00:00:00"

In [ ]:
# Visualizing Absolute Dynamic Topography (ADT) from Level-2 Data

import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# Geographical box

lon_min=-75
lon_max=-65
lat_min=30
lat_max=42

# Load the CSV dataset
df = pd.read_csv("jason1_l2_track_50.csv", comment='#')

# Select one cycle (e.g. 11)
df_cycle = df[df['cycle'] == ??]

fig = plt.figure(figsize=(16,8))
ax = plt.axes(projection=ccrs.PlateCarree(), aspect=1.25)
ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
ax.add_feature(cfeature.LAND, linewidth=1, facecolor='lightgrey', edgecolor='k', zorder=1)
ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True, linewidth=1, color='gray', alpha=0.5, linestyle='--')

# actually plot the data
tracks = ax.scatter(df_cycle['longitude'], df_cycle['latitude'], c=df_cycle['adt'],  vmin=-0.5, vmax=1.25, cmap='RdYlBu_r', marker='o', edgecolors='black', linewidth=0.01)
cbar = plt.colorbar(tracks)
plt.title('Dynamic Ocean Topography [m]', size=18, pad=24)
plt.show()

In [ ]:
# Visualizing the Real Satellite Orbital Drift (Zoom View)

# Because these are raw Level-2 observations, the ground tracks from different 
# repeat cycles do not overlap perfectly. By zooming in on a narrow 1-degree 
# latitude band, we can distinguish the orbital drift between consecutive cycles.

# NB. L3 are projected unto a "theoretical" track, thus for those products the 
# latitudes and longitudes for a same track are identical whatever the date/cycle

import pandas as pd
import matplotlib.pyplot as plt

# Load the dataset 
df = pd.read_csv("jason1_l2_track_50.csv", comment='#')

# Filter a very narrow latitudinal window 
zoom_df = df[(df['latitude'] >= 36.0) & (df['latitude'] <= 37.0)]

# Select the first 15 cycles available to keep the plot clean
unique_cycles = sorted(zoom_df['cycle'].unique())[:15]
zoom_subset = zoom_df[zoom_df['cycle'].isin(unique_cycles)]

# Create the scatter plot
plt.figure(figsize=(10, 8))

# Plot longitude on the X-axis and latitude on the Y-axis.
# Points are colored by cycle number to visually separate the physical satellite passes.
scatter = plt.scatter(zoom_subset['longitude'], zoom_subset['latitude'], c=zoom_subset['cycle'], cmap='viridis', s=10, edgecolors='k', linewidths=0.3)

# Configure plot layout, grid, and axis labels
plt.grid(True, linestyle='--', alpha=0.6)
plt.xlabel('Longitude [degrees_east]', size=12)
plt.ylabel('Latitude [degrees_north]', size=12)

cbar = plt.colorbar(scatter)
cbar.set_label('Satellite Cycle Number', size=12)

plt.title('Zoom on Jason-1 Pass 50: Orbital Drift Across Cycles', size=14, pad=15)
plt.show()

In [ ]:
# Grid Co-location using along-track BINs (Slide 8)

# Since repeat-pass observations do not occur at identical along-track positions,
# we define geographically fixed nominal points along the nominal track. 
# For each cycle, we associate the real Level-2 observations with their closest
# nominal point (binning) to align our time-series on a common grid.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Load the dataset 
df = pd.read_csv("jason1_l2_track_50.csv", comment='#')

# Define the nominal track
# Pass 50 is nearly straight, so we define a regular spacing in latitude.
lat_nominal = np.arange(30.0, 42.0, 0.05)
slope, intercept = np.polyfit(df['latitude'], df['longitude'], 1)
lon_nominal = slope * lat_nominal + intercept

# Store nominal coordinates as a reference array
nominal_points = np.column_stack((lon_nominal, lat_nominal))

# Binning: Assign each real L2 point to the nearest nominal point
binned_data = []

for cycle, df_cycle in df.groupby('cycle'):
    cycle_points = df_cycle[['longitude', 'latitude']].values
    
    # Calculate the distance matrix between all points in this cycle and the nominal points
    # (Using simple Euclidean distance since the area is small enough)
    distances = np.linalg.norm(cycle_points[:, np.newaxis, :] - nominal_points[np.newaxis, :, :], axis=2)
    
    # Find the index of the closest nominal point for each L2 observation
    closest_nominal_idx = np.argmin(distances, axis=1)
    
    # Check if the closest nominal point is within a reasonable threshold (e.g. ~10 km / 0.1 degrees)
    min_distances = np.min(distances, axis=1)
    valid_mask = min_distances < 0.1
    
    # Create a temporary DataFrame for the valid matched points
    matched_df = pd.DataFrame({
        'latitude': lat_nominal[closest_nominal_idx[valid_mask]],
        'longitude': lon_nominal[closest_nominal_idx[valid_mask]],
        'cycle': cycle,
        'adt': df_cycle['adt'].values[valid_mask],
        'time': df_cycle['time'].values[valid_mask]
    })
    
    # If multiple points in the same cycle fall into the same BIN, take the average
    cycle_binned = matched_df.groupby(['latitude', 'longitude', 'cycle'], as_index=False).agg({'adt': 'mean','time': 'first'})
    binned_data.append(cycle_binned)

# Combine into the final nominal track dataset
df_nominal = pd.concat(binned_data, ignore_index=True)
df_nominal.to_csv("jason1_binned_nominal.csv", index=False)

print(f"Co-location completed! Grouped L2 observations into {len(lat_nominal)} nominal along-track BINs.")

In [ ]:
# Same plot done before to visualize the drift among cycles but on the interpolated data

??

In [ ]:
# Here we show the seasonal variations of the Gulf Stream by grouping 
# the along-track observations into monthly profiles. 

# Map each cycle to its month of the year (1 = Jan, 12 = Dec)
df_nominal = pd.read_csv("jason1_binned_nominal.csv")

# Extract the month from the 'time' column
df_nominal['month'] = pd.to_datetime(df_nominal['time']).dt.month

fig = plt.figure(figsize=(8, 10))    
for month_id, df_month in df_nominal.groupby('month'):
    # Filter out bad data points
    df_month_filtered = df_month[df_month['adt'] > -0.5]
    averaged = df_month_filtered.groupby('latitude', as_index=False)['adt'].mean().sort_values(by='latitude')
    
    # Plot the monthly mean profile
    plt.plot(averaged['adt'], averaged['latitude'], marker='o', markersize=1.5, linestyle='-', linewidth=1, alpha=0.8, label=f"Month {month_id}")


# Plot styling
plt.title("Jason-1 Pass 50: Monthly Dynamic Ocean Topography Profiles", fontsize=14, pad=15)
plt.xlabel("Dynamic Ocean Topography [m]", fontsize=12)
plt.ylabel("Latitude [degrees_north]", fontsize=12)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(title="Months")
plt.show()

In [ ]:
# GEOSTROPHIC VELOCITY FOR: Single Track Profile vs. Avarage (Slide 17)

# To address the Slide 17 exercise, we compute the maximum geostrophic velocity (v_c)
# using two different approaches:
# 1) A SINGLE CYCLE (e.g., Cycle 18).
# 2) THE OVERALL MEAN.

import numpy as np
import pandas as pd

# Load dataset and clean bad values
df_nominal = pd.read_csv("jason1_binned_nominal.csv")
df_filtered = df_nominal[df_nominal['adt'] > -0.5]

# Constants
g = 9.81              # Gravity acceleration [m/s^2]
omega = 7.292115e-5   # Earth's angular velocity [rad/s]
R_earth = 6371000.0   # Earth's mean radius [m]

def compute_max_geostrophic_velocity(df_profile):
    df_sorted = df_profile.sort_values(by='latitude').reset_index(drop=True)
    
    lat_rad = np.radians(df_sorted['latitude'].values)
    lon_rad = np.radians(df_sorted['longitude'].values)
    
    # Cumulative along-track distance (s) in meters
    ds = R_earth * np.sqrt(np.diff(lat_rad)**2 + (np.diff(lon_rad) * np.cos(0.5 * (lat_rad[:-1] + lat_rad[1:])))**2)
    s = np.concatenate(([0], np.cumsum(ds)))
    
    # Coriolis parameter
    f = ??
    
    # Spatial gradient of Dynamic Ocean Topography (d_eta / ds)
    d_eta_ds = np.gradient(df_sorted['adt'].values, s)
    v_c = ??
    
    # Restrict search to 31°N to 41°N to avoid gradient-edge effects
    gs_mask = (df_sorted['latitude'] >= 31.0) & (df_sorted['latitude'] <= 41.0)
    v_c_gs = v_c[gs_mask]
    lats_gs = df_sorted['latitude'].values[gs_mask]
    
    max_idx = np.argmax(v_c_gs)
    return v_c_gs[max_idx], lats_gs[max_idx]

# APPROACH 1: Single Cycle Profile (e.g., Cycle 70)
# We pick a single pass 
cycle = ??
df_single_cycle = df_filtered[df_filtered['cycle'] == cycle]
max_v_single, lat_single = compute_max_geostrophic_velocity(df_single_cycle)

# APPROACH 2: Overall Climatological Mean
df_mean_profile = df_filtered.groupby('latitude', as_index=False).mean(numeric_only=True)
max_v_mean, lat_mean = ??

# Print the comparison
print(f"APPROACH 1: CYCLE {cycle}")
print(f"  -> Max Surface Velocity (v_c): {max_v_single:.2f} m/s")
print(f"  -> Gulf Stream Core Latitude : {lat_single:.2f}°N")
print("-" * 58)
print("APPROACH 2: AVERAGED CLIMATOLOGY")
print(f"  -> Max Surface Velocity (v_c): {max_v_mean:.2f} m/s")
print(f"  -> Gulf Stream Core Latitude : {lat_mean:.2f}°N")